In [ ]:
def solve_photo(filename, start, heading, goal, final_heading=None, show=True):
    """photo -> (command string, path, Maze). Raises if no route exists.

    Works for any rows x cols maze; both are measured from the photo.
    """
    img = cv2.imread(filename)
    if img is None:
        raise FileNotFoundError(filename)

    to_rig     = square_transform(find_rig_quad(img), COARSE_PX)
    coarse_img = cv2.warpPerspective(img, to_rig, (COARSE_PX, COARSE_PX))

    xs, ys = infer_grid(find_clips(coarse_img))
    cols, rows = len(xs) - 1, len(ys) - 1
    if rows < 2 or cols < 2:
        raise RuntimeError(f"implausible {rows} x {cols} grid from {filename}")

    quad = np.float32([[xs[0], ys[0]], [xs[-1], ys[0]], [xs[-1], ys[-1]], [xs[0], ys[-1]]])
    w, h = cols * CELL_PX, rows * CELL_PX
    to_grid = cv2.getPerspectiveTransform(
        quad, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    rect = cv2.warpPerspective(img, to_grid @ to_rig, (w, h))

    hw, vw, *_ = build_walls(wall_pixels(rect), rows, cols)
    m = Maze(hw, vw)

    for name, cell in (("start", start), ("goal", goal)):
        if not m.inside(*cell):
            raise ValueError(f"{name} {cell} is outside the {rows}x{cols} maze in {filename}")

    route = m.bfs(start, goal)
    if route is None:
        raise RuntimeError(f"no route from {start} to {goal} in {filename}")

    cmds, end = path_to_commands(route, heading, final_heading)
    landed, _ = replay(m, start, cmds, heading)
    assert landed == goal, f"replay ended at {landed}, expected {goal}"

    if show:
        print(f"{filename}: {rows} rows x {cols} cols")
        print(m.ascii(route, start, goal))
        print(f"\n{len(route) - 1} moves, facing {heading} -> {end}:  {''.join(cmds)}")
    return "".join(cmds), route, m


_ = solve_photo(PHOTO, START, START_HEADING, GOAL)

# photo_file = "maze_1.jpg"
# start_coord = (0, 3)
# start_orientation = "E" 
# goal_coord = (8, 5)
# solve_photo(photo_file, start_coord, start_orientation, goal_coord)